# 7. Porovnanie modelov

Všetky modely sú hodnotené rovnako: na rovnakých 10 deleniach train/test podľa zdroja.
Ako referenciu pridávame model, ktorý vždy predpovedá najčastejšiu triedu
v trénovacích dátach (úroveň náhody: balanced accuracy 0,5).
Výsledky sú zvlášť pre testovacie klipy z HMDB51 a z Charades (ak sú použité).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from har_utils import make_splits, split_metrics, summarize

df = pd.read_csv("index.csv")
y, ds = df.label.values, df.dataset.values

# referencia: vždy najčastejšia trieda z trénovacích dát
baseline = pd.DataFrame([
    r for seed, tr, te in make_splits(df)
    for r in split_metrics(seed, y[te], np.full(len(te), np.bincount(y[tr]).argmax()), ds[te])
])

files = {
    "CNN-SVM (ResNet18)": "results/cnn_svm.csv",
    "CNN-LSTM (ResNet18)": "results/cnn_lstm.csv",
    "R(2+1)D + SVM": "results/video_cnn_svm.csv",
    "R(2+1)D doladená": "results/video_finetune.csv",     # notebook 6b (voliteľný)
}
results = {"Najčastejšia trieda": baseline}
results |= {name: pd.read_csv(f) for name, f in files.items() if Path(f).exists()}

table = pd.concat([summarize(r, name) for name, r in results.items()])
table.columns = ["accuracy", "balanced accuracy", "F1 (macro)"]
table

In [ ]:
test_sets = list(dict.fromkeys(baseline.test_set))       # poradie: all, charades, hmdb
names = list(results)
fig, axes = plt.subplots(1, len(test_sets), figsize=(6 * len(test_sets), 4), squeeze=False)
for ax, t in zip(axes[0], test_sets):
    ax.boxplot([results[n].loc[results[n].test_set == t, "bal_acc"] for n in names],
               tick_labels=names)
    ax.axhline(0.5, ls="--", c="gray")
    ax.set_title(f"test: {t}")
    ax.set_ylabel("balanced accuracy (10 delení)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()